In [35]:
import torch.nn as nn
import torch

In [36]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0 / torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))

In [37]:
# 不会被依赖，仅用于展示如何在前向传播方法中添加快捷连接
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        # 五层深度神经网络，每层包含一个线性层和 GELU 激活函数
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU()),
        ])

    def forward(self, x):
        for layer in self.layers:
            layer_output = layer(x)
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x

## case1. 没有快捷连接

In [38]:
layer_sizes = [3,3,3,3,3,1]
torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(layer_sizes, False)

In [39]:
def print_gradients(model):
    input_ = torch.tensor([[1.,0.,-1.]])
    output = model(input_)
    target = torch.tensor([[0.]])

    loss = nn.MSELoss()
    loss = loss(output, target)

    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            print(name, param.grad.abs().mean().item())

In [40]:
print_gradients(model_without_shortcut)

layers.0.0.weight 0.00020173584925942123
layers.1.0.weight 0.00012011158833047375
layers.2.0.weight 0.0007152041071094573
layers.3.0.weight 0.0013988735154271126
layers.4.0.weight 0.005049645435065031


梯度在从最后一层（layers.4）到第一层（layers.0）时逐渐减小，这种现象称为梯度消失问题。

## case2 带有跳跃连接的模型

In [41]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(layer_sizes, True)
print_gradients(model_with_shortcut)

layers.0.0.weight 0.22169791162014008
layers.1.0.weight 0.20694106817245483
layers.2.0.weight 0.32896992564201355
layers.3.0.weight 0.2665732204914093
layers.4.0.weight 1.3258540630340576


从输出结果可以看到，最后一层（layers.4）的梯度依然比其他层更大。然而，随着接近第一层（layers.0），梯度值逐渐趋于稳定，并未缩小到几乎消失的程度。